# Experimento C — Outdoor vs. Indoor

## Preparación

In [4]:
%pip install pycolmap plotly open3d pandas tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Corrido en consola
# !sudo apt-get update -q
# !sudo apt-get install colmap -y -q

In [8]:
import os, shutil, time, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pycolmap
from pathlib import Path
from datetime import datetime

print("pycolmap:", pycolmap.__version__)


pycolmap: 4.0.4


### Paths e inicialización de carpetas

In [ ]:
BASE_DIR    = Path('./expC')
LOGS_DIR    = BASE_DIR / 'logs'
RESULTS_DIR = BASE_DIR / 'results'
SPARSE_DIR  = BASE_DIR / 'sparse'
DB_DIR      = BASE_DIR / 'db'
DATA_DIR    = Path('./data')
EXP_D_DIR   = Path('./expD')

for d in [LOGS_DIR, RESULTS_DIR, SPARSE_DIR, DB_DIR, EXP_D_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SCENES     = ['garden', 'bonsai']
SCENE_TYPE = {'garden': 'outdoor', 'bicycle': 'outdoor',
              'bonsai': 'indoor',  'counter': 'indoor'}

CSV_PATH = RESULTS_DIR / 'results_expC.csv'

SCALE = 4

def get_image_path(scene):
    for scale in [f'images_{SCALE}', 'images']:
        p = DATA_DIR / scene / scale
        if p.exists():
            return p
    raise FileNotFoundError(f"No se encontró carpeta de imágenes para {scene}")


### Funciones varias

Para guardar los logs en un txt. Así se monitorea el avance sin ensuciar el cuaderno.

In [ ]:
def execute_to_file(cmd, log_path, stage_name):
    """Ejecuta un comando y guarda stdout/stderr en el log."""
    print(f"  [>] {stage_name}...", end='', flush=True)
    t0 = time.time()
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(f"\n{'='*20}\nCOMANDO: {cmd}\n{'='*20}\n")
        proc = subprocess.Popen(cmd, shell=True, stdout=f,
                                stderr=subprocess.STDOUT, text=True)
        proc.wait()
    dt = time.time() - t0
    status = f"OK ({dt:.1f}s)" if proc.returncode == 0 else f"ERROR (ver {log_path.name})"
    print(f" {status}")
    return proc.returncode, dt

Para guardar los resultados en un mismo csv

In [ ]:
def append_to_csv(metrics, csv_path):
    """Agrega o actualiza una fila en el CSV acumulativo."""
    df_new = pd.DataFrame([metrics])
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df = pd.concat([df, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=['scene', 'config'], keep='last')
    else:
        df = df_new
    df.to_csv(csv_path, index=False)
    return df

Para contar componentes conexas

In [ ]:
def count_components(output_path):
    """Cuenta el número de componentes conexas generadas por COLMAP."""
    if not output_path.exists():
        return 0
    return len([d for d in output_path.iterdir()
                if d.is_dir() and d.name.isdigit()])

Para correr un pipeline COLMAP

Algunas definiciones y consideraciones

- `reco_path` es la ruta al componente conexo de mayor tamaño (el modelo principal) generado por el proceso de Sparse Reconstruction. COLMAP los enumera empezando por el 0 según la cantidad de imágenes registradas.
- objeto `reco`: carga a memoria RAM toda la información de un modelo 3D generado por COLMAP. Permite conectar las imágenes, cámaras y puntos 3D, calcular métricas de forma fácil usando sus métodos integrados, entre otros.
  - `reco.num_reg_images()`: cuenta cuantas imágenes tiene el modelo 3D. Sirve para evaluar cuantas de las imágenes que se le dieron al modelo fueron usadas.
  - `reco.compute_num_observations()`: calcula la suma de cuantas veces un punto 3D fue vistos por una imagen.
  - `reco.points3D`: los puntos que COLMAP logró reconstruir.
    - `reco.points3D.values()[x].track`: lista de todas las fotos donde aparece ese punto.

Como los archivos son pesados y el tiempo de ejecución es alto, se calcula la mayor cantidad de métricas posibles por ejecución, lo que no significa que se vayan a usar:
- scene, scene_type, config: contexto del epxerimento
- model: cámara usada
- total_images, images_registered, registration_ratio: registro de imágenes
- points_3d: cantidad de puntos reconstruidos
- mean_reprojection_error: promedio de error de ubicación de punto en la imagen y de ubicación caclulada.
- total_observations: cantidad de observaciones de todos los puntos en las imágenes
- obs_per_point: promedio de cámaras que ven a un mismo punto 3D
- avg_track_length: promedio de fotos en las que aparece cada punto
- std_track_length: dispersión de la visibilidad de los puntos
- median_track_length: mediana de de la visibilidad de los puntos
- max_track_length: el número máximo de fotos que comparten un mismo punto 3D
- num_connected_components: número de sub-modelos independientes generados
- execution_time_sec: tiempo de ejecución del proceso, en segundos
- sparse_path: ubicación de los archivos .bin o .txt resultantes
- images_path: ubicación de las fotos originales
- timestamp: fecha y hora de ejecución del pipeline


In [ ]:
def run_experiment(scene, conf):
    """
    Ejecuta el pipeline COLMAP completo para una escena y configuración.
    Guarda métricas en el CSV acumulativo y retorna el dict de métricas.
    """
    img_path  = get_image_path(scene)
    db_path   = DB_DIR      / f"{scene}_{conf['name']}.db"
    out_path  = SPARSE_DIR  / scene / conf['name']
    log_path  = LOGS_DIR    / f"log_{scene}_{conf['name']}.txt"

    out_path.mkdir(parents=True, exist_ok=True)
    
    if db_path.exists():  
        db_path.unlink()
    if log_path.exists(): 
        log_path.unlink()

    # Contar imágenes
    imgs_total = len(list(img_path.glob('*.jpg')) +
                     list(img_path.glob('*.JPG')) +
                     list(img_path.glob('*.png')))

    print(f"\n>>> {scene} | {conf['name']}  ({imgs_total} imágenes)")

    t_start = time.time()

    extract_cmd = (
        f"colmap feature_extractor "
        f"--database_path {db_path} "
        f"--image_path {img_path} "
        f"--ImageReader.single_camera 1 "
        f"--ImageReader.camera_model {conf['model']} "
        f"--SiftExtraction.use_gpu 1 "
        f"{conf.get('ext', '')}"
    )
    match_cmd = (
        f"colmap exhaustive_matcher "
        f"--database_path {db_path} "
        f"--SiftMatching.use_gpu 1 "
        f"{conf.get('match', '')}"
    )
    map_cmd = (
        f"colmap mapper "
        f"--database_path {db_path} "
        f"--image_path {img_path} "
        f"--output_path {out_path} "
        f"{conf.get('map', '')}"
    )

    execute_to_file(extract_cmd, log_path, "Extracción de features")
    execute_to_file(match_cmd,   log_path, "Matching")
    execute_to_file(map_cmd,     log_path, "Mapping (SfM)")

    t_end = time.time()

    reco_path    = out_path / "0"
    n_components = count_components(out_path)

    if reco_path.exists():
        reco  = pycolmap.Reconstruction(str(reco_path))
        imgs_reg = reco.num_reg_images()
        pts3d    = reco.num_points3D()
        n_obs    = reco.compute_num_observations()
        tracks   = (np.array([p.track.length() for p in reco.points3D.values()])
                    if pts3d > 0 else np.array([0]))

        metrics = {
            'scene'                     : scene,
            'scene_type'                : SCENE_TYPE[scene],
            'config'                    : conf['name'],
            'model'                     : conf['model'],
            'total_images'              : imgs_total,
            'images_registered'         : imgs_reg,
            'registration_ratio'        : imgs_reg / imgs_total if imgs_total > 0 else 0,
            'points_3d'                 : pts3d,
            'mean_reprojection_error'   : reco.compute_mean_reprojection_error(),
            'total_observations'        : n_obs,
            'obs_per_point'             : n_obs / pts3d if pts3d > 0 else 0,
            'avg_track_length'          : float(np.mean(tracks)),
            'std_track_length'          : float(np.std(tracks)),
            'median_track_length'       : float(np.median(tracks)),
            'max_track_length'          : float(np.max(tracks)),
            'num_connected_components'  : n_components,
            'execution_time_sec'        : t_end - t_start,
            'sparse_path'               : str(reco_path),
            'images_path'               : str(img_path),
            'timestamp'                 : datetime.now().isoformat(),
        }
    else:
        print(f"  [!] Sin reconstrucción generada para {conf['name']} en {scene}")
        metrics = {
            'scene'                     : scene,
            'scene_type'                : SCENE_TYPE[scene],
            'config'                    : conf['name'],
            'model'                     : conf['model'],
            'total_images'              : imgs_total,
            'images_registered'         : 0,
            'registration_ratio'        : 0.0,
            'points_3d'                 : 0,
            'mean_reprojection_error'   : None,
            'total_observations'        : 0,
            'obs_per_point'             : 0.0,
            'avg_track_length'          : 0.0,
            'std_track_length'          : 0.0,
            'median_track_length'       : 0.0,
            'max_track_length'          : 0.0,
            'num_connected_components'  : n_components,
            'execution_time_sec'        : t_end - t_start,
            'sparse_path'               : None,
            'images_path'               : str(img_path),
            'timestamp'                 : datetime.now().isoformat(),
        }

    append_to_csv(metrics, CSV_PATH)
    print(f"  Tiempo total: {t_end - t_start:.1f}s | "
          f"Registradas: {metrics['images_registered']} | "
          f"Puntos 3D: {metrics['points_3d']}")
    return metrics


## Verificación de imágenes disponibles

In [13]:
images_count = {}
for scene in SCENES:
    ip = get_image_path(scene)
    imgs = (list(ip.glob('*.jpg')) + list(ip.glob('*.JPG')) +
            list(ip.glob('*.png')))
    images_count[scene] = len(imgs)
    print(f"{scene:10s}: {len(imgs):4d} imágenes  ({ip})")


garden    :  185 imágenes  (data/garden/images)
bonsai    :  292 imágenes  (data/bonsai/images_4)


## Configuraciones ejecutadas

`Cam_OpenCV` y `Ext_HighFeatures`

Estas dos configuraciones se eligieron por ser las más informativas para la comparación:

- **`Cam_OpenCV`**: modelo de cámara con distorsión radial/tangencial (baseline realista).  
- **`Ext_HighFeatures`**: extracción con 8 000 features — evalúa si más descriptores mejoran la reconstrucción. Inicialmente se intentó con 12 000 y 10 000 features, pero esto superaba las capacidades de la máquina.

Ambas corren sobre `garden` (outdoor) y `bonsai` (indoor).


In [ ]:
CONFIGS_EXEC = [
    {
        "name"  : "Cam_OpenCV",
        "model" : "OPENCV",
        "ext"   : "",
        "match" : "",
        "map"   : "",
    },
    {
        "name"  : "Ext_HighFeatures",
        "model" : "OPENCV",
        "ext"   : "--SiftExtraction.max_num_features 8000",
        "match" : "",
        "map"   : "",
    },
]

results_exec = []
for scene in SCENES:
    for conf in CONFIGS_EXEC:
        if conf['name'] == "Cam_OpenCV":
            print(f"\n>>> Omitiendo {scene} con {conf['name']} (ya se corrió)")
            continue
        r = run_experiment(scene, conf)
        results_exec.append(r)

df_exec = pd.DataFrame(results_exec)
print("\n=== Resultados ejecutados ===")
# Algunas de las métricas
cols_show = ['scene', 'config', 'images_registered', 'registration_ratio',
             'points_3d', 'mean_reprojection_error', 'avg_track_length',
             'num_connected_components', 'execution_time_sec']
print(df_exec[cols_show].to_string(index=False))



>>> Omitiendo garden con Cam_OpenCV (ya se corrió)

>>> garden | Ext_HighFeatures  (185 imágenes)
  [>] Extracción de features... OK (74.1s)
  [>] Matching...

## Configuraciones previas

Las siguientes celdas contienen el código para otras configuraciones que se corrieron durante pruebas, que enriquecen el análisis, pero no sería provechoso correr de nuevo.

**No se ejecutan** por temas de tiempo


### Configuración 1

In [ ]:
CONFIGS_PREV_CAMMODELS = [
    {
        "name"  : "Cam_Pinhole",
        "model" : "PINHOLE",
        "ext"   : "",
        "match" : "",
        "map"   : "",
    },
]

# for scene in SCENES:
#     for conf in CONFIGS_PREV_CAMMODELS:
#         run_experiment(scene, conf)


### Configuración 2

In [ ]:
CONFIGS_PREV_EXT = [
    {
        "name"  : "Ext_LowFeatures",
        "model" : "OPENCV",
        "ext"   : "--SiftExtraction.max_num_features 4000",
        "match" : "",
        "map"   : "",
    },
]

# for scene in SCENES:
#     for conf in CONFIGS_PREV_EXT:
#         run_experiment(scene, conf)


### Configuración 3

In [ ]:
CONFIGS_PREV_MATCH = [
    {
        "name"  : "Match_Strict",
        "model" : "OPENCV",
        "ext"   : "",
        "match" : "--SiftMatching.max_error 1.5",
        "map"   : "",
    },
    {
        "name"  : "Match_Loose",
        "model" : "OPENCV",
        "ext"   : "",
        "match" : "--SiftMatching.max_error 6.0",
        "map"   : "",
    },
]

# for scene in SCENES:
#     for conf in CONFIGS_PREV_MATCH:
#         run_experiment(scene, conf)


### Configuración 4

In [ ]:
CONFIGS_PREV_MAP = [
    {
        "name"  : "Map_Robust",
        "model" : "OPENCV",
        "ext"   : "",
        "match" : "",
        "map"   : "--Mapper.min_model_size 10",
    },
]

# for scene in SCENES:
#     for conf in CONFIGS_PREV_MAP:
#         run_experiment(scene, conf)


# Resultados

## Resultados de configuraciones previas

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

In [ ]:
csv_path = f"{RESULTS_DIR}/results_prev_config.csv"

df = pd.read_csv(csv_path)

display(Markdown("---"))
display(Markdown("### Escena **Bonsai**"))
df_bonsai = df[df['scene'] == 'bonsai']
display(df_bonsai)

display(Markdown("---"))
display(Markdown("### Escena **Garden**"))
df_garden = df[df['scene'] == 'garden']

## Resultados de configuraciones que se corrieron

In [ ]:
csv_path = f"{RESULTS_DIR}/results_expC.csv"

df = pd.read_csv(csv_path)

display(Markdown("---"))
display(Markdown("### Escena **Bonsai**"))
df_bonsai = df[df['scene'] == 'bonsai']
display(df_bonsai)

display(Markdown("---"))
display(Markdown("### Escena **Garden**"))
df_garden = df[df['scene'] == 'garden']

# (a)

In [ ]:
METRICS = {
    'config': 'Configuración', 
    'images_registered': 'Imgs registradas',
    'registration_ratio': 'Ratio registro',
    'points_3d': 'Puntos 3D',
    'mean_reprojection_error': 'Error reproyección (px)',
    'avg_track_length': 'Track length promedio',
    'num_connected_components': 'Componentes conexas',
    'execution_time_sec': 'Tiempo ejecución (s)',
}

path_expC = f"{RESULTS_DIR}/results_expC.csv"
path_prev = f"{RESULTS_DIR}/results_prev_config.csv"

df_expC = pd.read_csv(path_expC)
df_prev = pd.read_csv(path_prev)

df_all = pd.concat([df_prev, df_expC], ignore_index=True)

def mostrar_tabla_escena(df, nombre_escena):
    display(Markdown(f"---"))
    display(Markdown(f"###Escena **{nombre_escena.capitalize()}**"))
    
    filtro = df[df['scene'] == nombre_escena.lower()][list(METRICS.keys())]
    
    filtro = filtro.rename(columns=METRICS)
    

mostrar_tabla_escena(df_all, "bonsai")
mostrar_tabla_escena(df_all, "garden")

# (b)

# (c)

# Exportar datos para Experimento D

In [ ]:
# Guardar paths de reconstrucciones y métricas base en ./expD/
exp_d_recs = df_all[['scene', 'config', 'model',
                      'sparse_path', 'images_path',
                      'images_registered', 'registration_ratio',
                      'points_3d', 'mean_reprojection_error',
                      'avg_track_length', 'execution_time_sec']].copy()

out_path_d = EXP_D_DIR / 'reconstructions_for_D.csv'
exp_d_recs.to_csv(out_path_d, index=False)
print(f"Guardado para D: {out_path_d}")
print(exp_d_recs[['scene','config','sparse_path']].to_string(index=False))
